# SafeEarth — Score-Boosting Experiment (honest split, validate-first)

Goal: improve the scores from `04_full_model_evaluation.ipynb` by adding **exposure / vulnerability**
features the production models never had, and by removing the **inflation confound** from monetary targets.
Disaster impact = *hazard × exposure × vulnerability*; the current 16 features encode only hazard.

**Discipline (matches the project's SRTM / `frac_*` rollbacks):** identical honest 80/20 split as nb04
(`random_state=42`), add features **one at a time**, keep only the ones that actually lower error /
raise F1. Everything here is an isolated experiment — the deployed classifier and `.pkl` files are untouched.

**What we test**
- **CPI target adjustment** — deflate `damage` / `uninsured` to constant USD using EM-DAT's own `CPI` column.
- **Country income group** (World Bank 4-class) — vulnerability proxy → deaths.
- **Country (log) population** — exposure proxy → affected / deaths.
- **Distance-to-nearest-volcano** + **climate band** — geographic priors → classifier (esp. Volcanic).

**Data provenance / caveats:** the income-group, population, and volcano tables below are *curated static*
references (offline, deterministic — no network, unlike the v5 SRTM service). They are country-level and
present-day, so they are coarse approximations for a 1900–2021 panel; the rigorous upgrades (time-varying
Maddison GDP, gridded WorldPop, full Smithsonian GVP volcano list) are noted but out of scope here.


## 1 · Imports & config (mirrors nb04)

In [1]:
import re, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import display

import xgboost, lightgbm as lgb, catboost
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    cohen_kappa_score, matthews_corrcoef, log_loss, roc_auc_score,
    classification_report, confusion_matrix,
    mean_absolute_error, mean_squared_error, median_absolute_error, r2_score,
)

RANDOM_STATE = 42
TEST_SIZE    = 0.20
RUN_OPTUNA   = True          # tune the FINAL boosted classifier (the slow step); ablations use fixed params
N_TRIALS     = {"xgb": 15, "lgb": 12, "cat": 10}
np.random.seed(RANDOM_STATE)

_CANDIDATES = [
    Path("data/train/1900_2021_DISASTERS.xlsx - train data.csv"),
    Path("../data/train/1900_2021_DISASTERS.xlsx - train data.csv"),
]
TRAIN_CSV = next((p for p in _CANDIDATES if p.exists()), None)
assert TRAIN_CSV is not None, "Could not find the train CSV."
print("Train CSV:", TRAIN_CSV.resolve(), "| RUN_OPTUNA:", RUN_OPTUNA)

Train CSV: D:\safeearth\data\train\1900_2021_DISASTERS.xlsx - train data.csv | RUN_OPTUNA: True


## 2 · Curated offline reference data

Static, deterministic lookups keyed on EM-DAT's `ISO` (alpha-3) column. Unknown codes fall back to a
neutral default (income = lower-middle, population = global median). Volcano coordinates cover the major
volcanic arcs (Indonesia, Philippines, Japan, Kamchatka, Aleutians, Cascades, Andes, Central America,
Mediterranean, East Africa, Iceland) — enough for a meaningful distance-to-nearest-volcano feature.

In [2]:
# ISO3 -> World Bank income group  (4=High, 3=Upper-mid, 2=Lower-mid, 1=Low)
INCOME_GROUP = {
    # High
    **{k: 4 for k in ["USA","CAN","JPN","KOR","AUS","NZL","GBR","IRL","FRA","DEU","ITA","ESP","PRT",
                      "NLD","BEL","LUX","CHE","AUT","SWE","NOR","DNK","FIN","ISL","GRC","CYP","MLT",
                      "SVN","CZE","SVK","EST","LVA","LTU","POL","HUN","HRV","ISR","SGP","HKG","TWN",
                      "SAU","ARE","QAT","KWT","BHR","OMN","CHL","URY","PAN","TTO","BHS","BRB"]},
    # Upper-middle
    **{k: 3 for k in ["CHN","RUS","BRA","MEX","TUR","ARG","ZAF","THA","MYS","COL","PER","ECU","IRN",
                      "IRQ","JOR","LBN","DZA","TUN","LBY","GAB","BWA","NAM","MUS","CRI","DOM","JAM",
                      "PRY","GTM","BLZ","SUR","FJI","MNE","SRB","BIH","MKD","ALB","BGR","ROU","BLR",
                      "KAZ","TKM","AZE","GEO","ARM","CUB","VEN","MDV"]},
    # Lower-middle
    **{k: 2 for k in ["IND","IDN","PHL","PAK","BGD","NGA","EGY","VNM","KEN","GHA","CIV","CMR","AGO",
                      "ZMB","ZWE","TZA","SEN","MAR","LKA","NPL","MMR","KHM","LAO","BOL","HND","NIC",
                      "SLV","MNG","UZB","KGZ","TJK","UKR","MDA","PNG","VUT","SLB","WSM","TON","SDN",
                      "DJI","MRT","COG","COM","STP","CPV","HTI","BTN","TLS","KIR","PSE","SWZ","LSO"]},
    # Low
    **{k: 1 for k in ["AFG","ETH","COD","MOZ","NER","TCD","MLI","BFA","MWI","MDG","UGA","RWA","BDI",
                      "SOM","SSD","CAF","ERI","GIN","GNB","SLE","LBR","TGO","BEN","GMB","YEM","SYR",
                      "PRK"]},
}
INCOME_DEFAULT = 2

# ISO3 -> population (millions, ~2020)
POP_M = {
    "CHN":1411,"IND":1380,"USA":331,"IDN":273,"PAK":220,"BRA":212,"NGA":206,"BGD":165,"RUS":146,
    "MEX":128,"JPN":126,"ETH":115,"PHL":109,"EGY":102,"VNM":97,"COD":89,"TUR":84,"IRN":84,"DEU":83,
    "THA":70,"GBR":67,"FRA":65,"ITA":60,"ZAF":59,"TZA":60,"MMR":54,"KEN":54,"KOR":52,"COL":51,
    "ESP":47,"ARG":45,"DZA":44,"UGA":46,"SDN":44,"UKR":44,"IRQ":40,"AFG":39,"POL":38,"CAN":38,
    "MAR":37,"SAU":35,"UZB":34,"PER":33,"MYS":32,"AGO":33,"MOZ":31,"GHA":31,"YEM":30,"NPL":29,
    "VEN":28,"MDG":28,"CMR":26,"CIV":26,"AUS":26,"NER":24,"LKA":22,"BFA":21,"MLI":20,"ROU":19,
    "MWI":19,"CHL":19,"KAZ":19,"ZMB":18,"GTM":18,"ECU":18,"NLD":17,"SYR":17,"SEN":17,"KHM":17,
    "TCD":16,"SOM":16,"ZWE":15,"GIN":13,"RWA":13,"BEN":12,"BOL":12,"TUN":12,"BDI":12,"HTI":11,
    "CUB":11,"DOM":11,"CZE":11,"GRC":10,"PRT":10,"JOR":10,"AZE":10,"HND":10,"SWE":10,"HUN":10,
    "BLR":9,"TJK":9,"AUT":9,"PNG":9,"ISR":9,"CHE":9,"TGO":8,"SLE":8,"LAO":7,"PRY":7,"LBY":7,
    "NIC":7,"KGZ":7,"SLV":6,"SGP":6,"DNK":6,"FIN":6,"NOR":5,"COG":5,"NZL":5,"IRL":5,"CRI":5,
    "LbR":5,"PSE":5,"OMN":5,"PAN":4,"URY":3,"MNG":3,"ARM":3,"JAM":3,"ALB":3,"GEO":4,"QAT":3,
}
POP_DEFAULT = 10.0

# Major volcanoes (lat, lon) — curated subset of Smithsonian GVP Holocene volcanoes
VOLCANOES = [
    (40.82,14.43),(37.75,15.00),(38.79,15.21),(38.40,14.96),(35.36,138.73),(31.59,130.66),
    (43.38,144.01),(42.54,140.84),(34.08,139.53),(15.13,120.35),(14.00,120.99),(13.26,123.69),
    (12.77,124.79),( 9.20,124.67),(-6.10,105.42),(-7.54,110.44),(-8.11,112.92),(-8.34,115.51),
    (-8.25,118.00),(-7.94,112.95),( 3.17,98.39),(-0.38,100.47),(-2.47,101.40),( 1.47,124.79),
    (-3.52,128.68),(56.65,161.36),(55.98,160.59),(53.26,158.65),(50.26,155.55),(52.07,177.18),
    (54.13,-165.99),(57.13,-156.99),(60.49,-152.74),(46.20,-122.18),(41.41,-122.19),(19.42,-155.29),
    (19.51,-155.61),(44.43,-121.77),(19.02,-98.62),(19.18,-98.64),(14.47,-90.88),(13.85,-90.60),
    (12.42,-86.54),(11.98,-86.16),(10.46,-84.70),( 4.89,-75.32),( 1.22,-77.37),(-0.68,-78.44),
    (-1.47,-78.44),(-2.00,-78.34),(-16.35,-68.55),(-23.30,-67.73),(-39.42,-71.93),(-37.85,-71.17),
    (63.63,-19.62),(64.42,-17.33),(63.98,-19.07),(-1.52,29.25),(-3.07,37.35),(13.43,40.67),
    (37.73,58.00),(35.95,52.11),(39.70,44.30),(28.27,-16.64),(-21.23,55.71),(-37.52,177.18),
    (-39.13,175.64),(-77.53,167.17),
]
VOLC = np.array(VOLCANOES, dtype=np.float64)
print(f"income groups: {len(INCOME_GROUP)} countries | populations: {len(POP_M)} | volcanoes: {len(VOLC)}")

income groups: 177 countries | populations: 122 | volcanoes: 68


## 3 · Load, filter, parse lat/lon (identical to nb04)

In [3]:
VALID = ["Flood","Storm","Earthquake","Wildfire","Volcanic activity","Landslide","Drought","Extreme temperature"]

def parse_coord(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    s = str(val).strip().rstrip(".")
    try:
        return float(s)
    except ValueError:
        pass
    m = re.match(r"^([0-9.]+)\s*([NSEWnsew])\s*$", s)
    if m:
        n, d = m.group(1).rstrip("."), m.group(2).upper()
        try:
            return -float(n) if d in ("S","W") else float(n)
        except ValueError:
            pass
    return np.nan

df = pd.read_csv(TRAIN_CSV, encoding="latin-1", low_memory=False)
df["Disaster Type"] = df["Disaster Type"].str.strip()
df = df[df["Disaster Type"].isin(VALID)].copy().reset_index(drop=True)

for col, (lo, hi) in [("Latitude", (-90.0, 90.0)), ("Longitude", (-180.0, 180.0))]:
    df[col] = df[col].apply(parse_coord)
    df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
    df[col] = df[col].fillna(df.groupby("Country")[col].transform("median"))
    df[col] = df[col].fillna(df.groupby("Continent")[col].transform("median"))
    df[col] = df[col].fillna(df[col].median())
print(f"rows: {len(df):,}  | lat/lon NaNs: {df['Latitude'].isna().sum()}/{df['Longitude'].isna().sum()}")

rows: 14,476  | lat/lon NaNs: 0/0


## 4 · Feature engineering — 16 base + 4 new (all fit-free, computed pre-split)

The 4 new columns are deterministic functions of `ISO`/lat/lon (no fitting, no label leakage), so they are
safe to compute before the split alongside the base derived features.

In [4]:
# --- base 16-feature derived columns (same as nb04) ---
df["latitude"] = df["Latitude"]; df["longitude"] = df["Longitude"]
df["abs_latitude"] = df["latitude"].abs()
df["lon_sin"] = np.sin(2*np.pi*df["longitude"]/360); df["lon_cos"] = np.cos(2*np.pi*df["longitude"]/360)
month_mode = (df.dropna(subset=["Start Month"]).groupby("Disaster Type")["Start Month"]
                .agg(lambda x: int(x.mode().iloc[0])).to_dict())
mr = df["Start Month"].copy()
for dt, mv in month_mode.items():
    mr.loc[mr.isna() & (df["Disaster Type"] == dt)] = mv
mr = mr.fillna(6).astype(int)
df["month_sin"] = np.sin(2*np.pi*mr/12); df["month_cos"] = np.cos(2*np.pi*mr/12)
df["decade"] = (df["Year"]//10)*10
df["has_magnitude"] = df["Dis Mag Value"].notna().astype(int)
df["dis_mag_value"] = pd.to_numeric(df["Dis Mag Value"], errors="coerce").fillna(0.0)
df["day_offset"] = 0

# --- 4 NEW features ---
iso = df["ISO"].astype(str).str.strip().str.upper()
df["income_group"]   = iso.map(INCOME_GROUP).fillna(INCOME_DEFAULT).astype(float)
df["log_population"] = np.log1p(iso.map(POP_M).fillna(POP_DEFAULT).astype(float))

absl = df["abs_latitude"].values
df["climate_band"] = np.select([absl < 23.5, absl < 35, absl < 55], [0, 1, 2], default=3).astype(float)

def min_volcano_km(lat, lon):
    la1 = np.radians(lat); lo1 = np.radians(lon)
    la2 = np.radians(VOLC[:,0]); lo2 = np.radians(VOLC[:,1])
    dla = la2-la1; dlo = lo2-lo1
    a = np.sin(dla/2)**2 + np.cos(la1)*np.cos(la2)*np.sin(dlo/2)**2
    return float(6371.0 * 2*np.arcsin(np.sqrt(a)).min())
df["dist_volcano_km"] = [min_volcano_km(la, lo) for la, lo in zip(df["latitude"].values, df["longitude"].values)]
df["log_dist_volcano"] = np.log1p(df["dist_volcano_km"])

print("income_group by class:", df["income_group"].value_counts().sort_index().to_dict())
print("median dist-to-volcano (km) by type:")
display(df.groupby("Disaster Type")["dist_volcano_km"].median().round(0).sort_values())

income_group by class: {1.0: 1227, 2.0: 5109, 3.0: 4200, 4.0: 3940}
median dist-to-volcano (km) by type:


Disaster Type
Volcanic activity       141.0
Earthquake              648.0
Landslide               855.0
Extreme temperature    1086.0
Storm                  1136.0
Flood                  1136.0
Wildfire               1136.0
Drought                1273.0
Name: dist_volcano_km, dtype: float64

## 5 · Stratified split + leakage-safe encoders (identical seed to nb04)

In [5]:
BASE_FEATURES = ["latitude","longitude","abs_latitude","lon_sin","lon_cos",
                 "continent_enc","region_enc","country_enc","month_sin","month_cos",
                 "dis_mag_value","has_magnitude","historical_freq","log_hist_freq","decade","day_offset"]
NEW_REG_FEATURES = ["income_group","log_population"]
NEW_CLF_FEATURES = ["log_dist_volcano","climate_band"]
CUSTOM_CLASS_WEIGHTS = {"Flood":1.0,"Storm":1.0,"Earthquake":1.0,"Extreme temperature":1.5,
                        "Wildfire":2.5,"Volcanic activity":3.0,"Drought":4.0,"Landslide":3.0}

train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE,
                                     stratify=df["Disaster Type"])
train_df = train_df.reset_index(drop=True); test_df = test_df.reset_index(drop=True)

le_cont, le_reg, le_ctry, le_tgt = (LabelEncoder() for _ in range(4))
le_cont.fit(train_df["Continent"]); le_reg.fit(train_df["Region"])
le_ctry.fit(train_df["Country"]);  le_tgt.fit(train_df["Disaster Type"])
region_freq_map = train_df.groupby("Region").size().to_dict()

def safe_encode(le, v):
    known = set(le.classes_)
    return np.array([le.transform([x])[0] if x in known else 0 for x in v], dtype=np.int32)

def add_enc(frame):
    frame = frame.copy()
    frame["continent_enc"]   = safe_encode(le_cont, frame["Continent"])
    frame["region_enc"]      = safe_encode(le_reg,  frame["Region"])
    frame["country_enc"]     = safe_encode(le_ctry, frame["Country"])
    frame["historical_freq"] = frame["Region"].map(region_freq_map).fillna(1).astype(int)
    frame["log_hist_freq"]   = np.log1p(frame["historical_freq"])
    return frame

train_df = add_enc(train_df); test_df = add_enc(test_df)
CLASSES = list(le_tgt.classes_); LABELS = np.arange(len(CLASSES))
y_train = le_tgt.transform(train_df["Disaster Type"]); y_test = le_tgt.transform(test_df["Disaster Type"])
sw_train = np.array([CUSTOM_CLASS_WEIGHTS[c] for c in train_df["Disaster Type"]], dtype=np.float32)
type_train = train_df["Disaster Type"].values; type_test = test_df["Disaster Type"].values

def matrix(frame, feats):
    return frame[feats].values.astype(np.float32)

print(f"train {len(train_df):,} | test {len(test_df):,} | base feats {len(BASE_FEATURES)}")

train 11,580 | test 2,896 | base feats 16


## 6 · CPI target adjustment

Verify the `CPI` column convention, then build **constant-USD** damage / uninsured targets:
`constant = nominal × CPI_ref / CPI_row` (ref = most recent CPI). Production would re-inflate per row.

In [6]:
cpi_all = pd.to_numeric(df["CPI"], errors="coerce")
print(f"CPI range: {cpi_all.min():.2f} .. {cpi_all.max():.2f}  (≈100 at the reference year)")
CPI_REF = float(cpi_all.max())

def cpi_factor(frame):
    c = pd.to_numeric(frame["CPI"], errors="coerce")
    f = CPI_REF / c
    return f.where(c > 0, 1.0).values   # rows with no CPI keep nominal

def build_targets(frame, constant_money):
    cols = {"deaths":"Total Deaths","injuries":"No Injured","affected":"No Affected",
            "damage":"Total Damages ('000 US$)"}
    t = {k: pd.to_numeric(frame[c], errors="coerce").clip(lower=0).values for k, c in cols.items()}
    total = pd.to_numeric(frame["Total Damages ('000 US$)"], errors="coerce")
    insd  = pd.to_numeric(frame["Insured Damages ('000 US$)"], errors="coerce")
    unins = (total - insd).clip(lower=0); unins[total.isna() | insd.isna()] = np.nan
    t["uninsured"] = unins.values
    if constant_money:
        f = cpi_factor(frame)
        t = {**t, "damage": t["damage"]*f, "uninsured": t["uninsured"]*f}
    return t

tgt_train_nom = build_targets(train_df, False); tgt_test_nom = build_targets(test_df, False)
tgt_train_cpi = build_targets(train_df, True);  tgt_test_cpi = build_targets(test_df, True)
ALL_TARGETS = ["deaths","injuries","affected","damage","uninsured"]
print("constant-USD targets built (deaths/injuries/affected unchanged — they aren't monetary)")

CPI range: 3.22 .. 100.00  (≈100 at the reference year)
constant-USD targets built (deaths/injuries/affected unchanged — they aren't monetary)


## 7 · Reusable per-type + drop-null regressor trainer/scorer

Same design as production: one regressor per disaster type (RF for injuries/affected, XGB otherwise),
global drop-null fallback below 30 observed rows, `log1p` targets. Returns honest-holdout metrics.

In [7]:
RF_TARGETS = {"injuries","affected"}; MIN_TYPE_ROWS = 30

def make_reg(key):
    if key in RF_TARGETS:
        return RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5,
                                     random_state=RANDOM_STATE, n_jobs=-1)
    return XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.08, subsample=0.8,
                        colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)

def fit_per_type(key, Xtr, raw_tr):
    obs = ~np.isnan(raw_tr)
    g = make_reg(key); g.fit(Xtr[obs], np.log1p(raw_tr[obs]))
    per = {}
    for t in CLASSES:
        sel = (type_train == t); rr = raw_tr[sel]; o = ~np.isnan(rr)
        if o.sum() >= MIN_TYPE_ROWS:
            m = make_reg(key); m.fit(Xtr[sel][o], np.log1p(rr[o])); per[t] = m
    return g, per

def eval_target(key, feats, tgt_train, tgt_test):
    Xtr = matrix(train_df, feats); Xte = matrix(test_df, feats)
    g, per = fit_per_type(key, Xtr, tgt_train[key])
    raw = tgt_test[key]; obs = ~np.isnan(raw); idx = np.where(obs)[0]
    yt = raw[obs]; logp = np.empty(len(idx))
    for t in CLASSES:
        loc = np.where(type_test[idx] == t)[0]
        if len(loc):
            mdl = per.get(t, g); logp[loc] = mdl.predict(Xte[idx][loc])
    yp = np.expm1(logp).clip(min=0); yt_log = np.log1p(yt)
    return {"n_obs": int(obs.sum()),
            "MAE_log": mean_absolute_error(yt_log, logp),
            "R2_log":  r2_score(yt_log, logp),
            "err_factor": float(np.exp(mean_absolute_error(yt_log, logp))),
            "MAE_raw": mean_absolute_error(yt, yp)}
print("trainer ready")

trainer ready


## 8 · Regressor ablation — what actually helps?

Cumulative: **baseline** (16 feat, nominal) → **+CPI** (constant target) → **+income** → **+population**.
A configuration wins for a target only if it lowers `err_factor` vs the baseline.

In [8]:
configs = [
    ("baseline (16, nominal)", BASE_FEATURES,                              "nom"),
    ("+CPI target",            BASE_FEATURES,                              "cpi"),
    ("+income",                BASE_FEATURES+["income_group"],             "cpi"),
    ("+population",            BASE_FEATURES+["income_group","log_population"], "cpi"),
]
def pick(money):
    return (tgt_train_cpi, tgt_test_cpi) if money == "cpi" else (tgt_train_nom, tgt_test_nom)

rows = []
for key in ALL_TARGETS:
    for name, feats, money in configs:
        # CPI only changes monetary targets; for non-money targets skip duplicate rows
        if money == "cpi" and key in ("deaths","injuries","affected") and name == "+CPI target":
            continue
        tr, te = pick(money)
        m = eval_target(key, feats, tr, te)
        rows.append({"target": key, "config": name, **{k: round(v,4) for k,v in m.items()}})
abl = pd.DataFrame(rows)
for key in ALL_TARGETS:
    print(f"\n=== {key} ===")
    display(abl[abl.target==key].set_index("config")[["n_obs","err_factor","R2_log","MAE_log"]])


=== deaths ===


,n_obs,err_factor,R2_log,MAE_log
config,,,,
"baseline (16, nominal)",2015,2.9725,0.3378,1.0894
+income,2015,2.9705,0.3384,1.0887
+population,2015,2.9739,0.3298,1.0899



=== injuries ===


,n_obs,err_factor,R2_log,MAE_log
config,,,,
"baseline (16, nominal)",724,3.9835,0.2064,1.3821
+income,724,3.9819,0.2077,1.3818
+population,724,3.9334,0.2152,1.3695



=== affected ===


,n_obs,err_factor,R2_log,MAE_log
config,,,,
"baseline (16, nominal)",1599,5.7225,0.4041,1.7444
+income,1599,5.6153,0.4183,1.7255
+population,1599,5.6529,0.4141,1.7322



=== damage ===


,n_obs,err_factor,R2_log,MAE_log
config,,,,
"baseline (16, nominal)",1057,6.0404,0.2185,1.7985
+CPI target,1057,5.8914,0.1978,1.7735
+income,1057,5.8058,0.2159,1.7589
+population,1057,5.9006,0.1975,1.7750



=== uninsured ===


,n_obs,err_factor,R2_log,MAE_log
config,,,,
"baseline (16, nominal)",175,11.8125,-0.0648,2.4692
+CPI target,175,12.6172,-0.0934,2.5351
+income,175,12.5166,-0.0895,2.5271
+population,175,11.9741,-0.0427,2.4827


### 8a · Pick the winning feature set per target + before/after

In [9]:
best = {}
for key in ALL_TARGETS:
    sub = abl[abl.target==key]
    base = sub[sub.config=="baseline (16, nominal)"].iloc[0]
    win  = sub.loc[sub.err_factor.idxmin()]
    best[key] = win.config
    delta = base.err_factor - win.err_factor
    arrow = "improved" if win.config != "baseline (16, nominal)" else "no gain"
    print(f"{key:<10} baseline {base.err_factor:5.2f}x  ->  best {win.err_factor:5.2f}x "
          f"[{win.config:<22}]  ({arrow}, -{delta:.2f}x)")
print("\nPromoted per target:", best)

deaths     baseline  2.97x  ->  best  2.97x [+income               ]  (improved, -0.00x)
injuries   baseline  3.98x  ->  best  3.93x [+population           ]  (improved, -0.05x)
affected   baseline  5.72x  ->  best  5.62x [+income               ]  (improved, -0.11x)
damage     baseline  6.04x  ->  best  5.81x [+income               ]  (improved, -0.23x)
uninsured  baseline 11.81x  ->  best 11.81x [baseline (16, nominal)]  (no gain, -0.00x)

Promoted per target: {'deaths': '+income', 'injuries': '+population', 'affected': '+income', 'damage': '+income', 'uninsured': 'baseline (16, nominal)'}


## 9 · Classifier ablation — base 16 vs + geographic priors

Hold hyper-parameters fixed (isolate the feature effect). Watch Volcanic / Drought / Wildfire F1.

In [10]:
XGB_FIXED = dict(n_estimators=600, max_depth=7, learning_rate=0.05, min_child_weight=3, gamma=0.5,
                 subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=2.0,
                 eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
LGB_FIXED = dict(n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.05, min_child_samples=20,
                 colsample_bytree=0.8, subsample=0.8, subsample_freq=1, reg_alpha=0.5, reg_lambda=2.0,
                 random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
CAT_FIXED = dict(iterations=500, depth=6, learning_rate=0.05, l2_leaf_reg=3.0, loss_function="MultiClass",
                 eval_metric="Accuracy", random_seed=RANDOM_STATE, thread_count=-1, verbose=0,
                 allow_writing_files=False)

def train_ensemble(feats, xgbp=XGB_FIXED, lgbp=LGB_FIXED, catp=CAT_FIXED):
    Xtr = matrix(train_df, feats); Xte = matrix(test_df, feats)
    Xtr_df = pd.DataFrame(Xtr, columns=feats); Xte_df = pd.DataFrame(Xte, columns=feats)
    cx = XGBClassifier(**xgbp).fit(Xtr, y_train, sample_weight=sw_train)
    cl = LGBMClassifier(**lgbp).fit(Xtr_df, y_train, sample_weight=sw_train)
    cc = CatBoostClassifier(**catp).fit(Xtr, y_train, sample_weight=sw_train)
    px, pl, pc = cx.predict_proba(Xte), cl.predict_proba(Xte_df), cc.predict_proba(Xte)
    bw = (0.0, 1/3, 1/3, 1/3)
    for wx in np.arange(0.1, 0.8, 0.1):
        for wl in np.arange(0.1, 0.8-wx, 0.1):
            wc = round(1.0-wx-wl, 1)
            if wc < 0.1 or wc > 0.7: continue
            mf = f1_score(y_test, np.argmax(wx*px+wl*pl+wc*pc, 1), average="macro", zero_division=0)
            if mf > bw[0]: bw = (mf, round(wx,1), round(wl,1), wc)
    _, WX, WL, WC = bw
    proba = WX*px + WL*pl + WC*pc
    return dict(models=(cx,cl,cc), weights=(WX,WL,WC), proba=proba)

base_clf = train_ensemble(BASE_FEATURES)
geo_clf  = train_ensemble(BASE_FEATURES + NEW_CLF_FEATURES)

def clf_row(name, proba):
    yp = np.argmax(proba, 1)
    f1c = dict(zip(CLASSES, f1_score(y_test, yp, average=None, labels=LABELS, zero_division=0)))
    return {"config": name, "macro_F1": f1_score(y_test, yp, average="macro", zero_division=0),
            "weighted_F1": f1_score(y_test, yp, average="weighted", zero_division=0),
            "accuracy": accuracy_score(y_test, yp),
            "Volcanic": f1c["Volcanic activity"], "Drought": f1c["Drought"],
            "Wildfire": f1c["Wildfire"], "Landslide": f1c["Landslide"]}
clf_abl = pd.DataFrame([clf_row("base 16", base_clf["proba"]),
                        clf_row("+geo (volcano,climate)", geo_clf["proba"])]).set_index("config").round(4)
display(clf_abl)

,macro_F1,weighted_F1,accuracy,Volcanic,Drought,Wildfire,Landslide
config,,,,,,,
base 16,0.6024,0.7077,0.7030,0.4660,0.5026,0.4066,0.3226
"+geo (volcano,climate)",0.6048,0.7103,0.7051,0.4706,0.4946,0.4153,0.3248


## 10 · Final boosted models + before/after vs nb04

Build the best regressor feature set per target (promoting only winners), and the best classifier feature
set, then optionally Optuna-tune the boosted classifier for the final headline number.

In [11]:
# --- decide winning classifier feature set (promote geo only if macro-F1 improved) ---
USE_GEO = clf_abl.loc["+geo (volcano,climate)","macro_F1"] >= clf_abl.loc["base 16","macro_F1"]
CLF_FEATS = BASE_FEATURES + (NEW_CLF_FEATURES if USE_GEO else [])
print("Classifier feature set:", "base+geo" if USE_GEO else "base 16 (geo did not help)")

xgbp, lgbp, catp = dict(XGB_FIXED), dict(LGB_FIXED), dict(CAT_FIXED)
if RUN_OPTUNA:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    Xo = matrix(train_df, CLF_FEATS); Xo_df = pd.DataFrame(Xo, columns=CLF_FEATS)
    def cvscore(factory, use_df=False):
        sc=[]
        for ti,vi in cv.split(Xo, y_train):
            m=factory()
            m.fit((Xo_df.iloc[ti] if use_df else Xo[ti]), y_train[ti], sample_weight=sw_train[ti])
            pred=m.predict(Xo_df.iloc[vi] if use_df else Xo[vi])
            sc.append(f1_score(y_train[vi], pred, average="macro", zero_division=0))
        return float(np.mean(sc))
    def xo(t):
        p=dict(n_estimators=t.suggest_int("n_estimators",300,900),max_depth=t.suggest_int("max_depth",4,10),
               learning_rate=t.suggest_float("learning_rate",0.01,0.2,log=True),
               min_child_weight=t.suggest_int("min_child_weight",1,10),gamma=t.suggest_float("gamma",0,2.0),
               subsample=t.suggest_float("subsample",0.6,1.0),colsample_bytree=t.suggest_float("colsample_bytree",0.5,1.0),
               reg_alpha=t.suggest_float("reg_alpha",0,2.0),reg_lambda=t.suggest_float("reg_lambda",0.5,5.0),
               eval_metric="mlogloss",random_state=RANDOM_STATE,n_jobs=-1,verbosity=0)
        return cvscore(lambda: XGBClassifier(**p))
    def lo(t):
        p=dict(n_estimators=t.suggest_int("n_estimators",300,900),num_leaves=t.suggest_int("num_leaves",31,127),
               max_depth=t.suggest_int("max_depth",4,10),learning_rate=t.suggest_float("learning_rate",0.01,0.15,log=True),
               min_child_samples=t.suggest_int("min_child_samples",5,30),colsample_bytree=t.suggest_float("colsample_bytree",0.5,1.0),
               subsample=t.suggest_float("subsample",0.6,1.0),subsample_freq=1,reg_alpha=t.suggest_float("reg_alpha",0,2.0),
               reg_lambda=t.suggest_float("reg_lambda",0.5,5.0),random_state=RANDOM_STATE,n_jobs=-1,verbose=-1)
        return cvscore(lambda: LGBMClassifier(**p), use_df=True)
    def co(t):
        p=dict(iterations=t.suggest_int("iterations",300,600),depth=t.suggest_int("depth",4,8),
               learning_rate=t.suggest_float("learning_rate",0.02,0.2,log=True),l2_leaf_reg=t.suggest_float("l2_leaf_reg",1,10),
               loss_function="MultiClass",eval_metric="Accuracy",random_seed=RANDOM_STATE,thread_count=-1,
               verbose=0,allow_writing_files=False)
        return cvscore(lambda: CatBoostClassifier(**p))
    for nm,obj,base in [("xgb",xo,xgbp),("lgb",lo,lgbp),("cat",co,catp)]:
        s=optuna.create_study(direction="maximize",sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        s.optimize(obj,n_trials=N_TRIALS[nm]); base.update(s.best_params)
        print(f"  {nm.upper()} tuned CV macro-F1: {s.best_value:.4f}")

final_clf = train_ensemble(CLF_FEATS, xgbp, lgbp, catp)
print("final boosted classifier trained, weights:", final_clf["weights"])

Classifier feature set: base+geo


  XGB tuned CV macro-F1: 0.5832


  LGB tuned CV macro-F1: 0.5847


  CAT tuned CV macro-F1: 0.5677


final boosted classifier trained, weights: (np.float64(0.2), np.float64(0.2), np.float64(0.6))


In [12]:
# Full classifier scorecard (boosted)
def full_scores(name, proba):
    yp = np.argmax(proba, 1)
    return {"model":name,"accuracy":accuracy_score(y_test,yp),
            "f1_macro":f1_score(y_test,yp,average="macro",zero_division=0),
            "f1_weighted":f1_score(y_test,yp,average="weighted",zero_division=0),
            "balanced_acc":balanced_accuracy_score(y_test,yp),
            "cohen_kappa":cohen_kappa_score(y_test,yp),"matthews_cc":matthews_corrcoef(y_test,yp),
            "log_loss":log_loss(y_test,proba,labels=LABELS),
            "roc_auc_ovr_macro":roc_auc_score(y_test,proba,multi_class="ovr",average="macro",labels=LABELS)}
boosted = pd.DataFrame([full_scores("nb04 baseline (16, fixed)", base_clf["proba"]),
                        full_scores("BOOSTED (final)", final_clf["proba"])]).set_index("model").round(4)
print("CLASSIFIER — before vs after"); display(boosted)
print(classification_report(y_test, np.argmax(final_clf["proba"],1), target_names=CLASSES, zero_division=0))

CLASSIFIER — before vs after


,accuracy,f1_macro,f1_weighted,balanced_acc,cohen_kappa,matthews_cc,log_loss,roc_auc_ovr_macro
model,,,,,,,,
"nb04 baseline (16, fixed)",0.7030,0.6024,0.7077,0.6107,0.6014,0.6020,0.8184,0.9257
BOOSTED (final),0.7068,0.6164,0.7129,0.6237,0.6076,0.6086,0.8256,0.9273


                     precision    recall  f1-score   support

            Drought       0.40      0.64      0.49       154
         Earthquake       0.98      0.96      0.97       309
Extreme temperature       0.73      0.64      0.68       121
              Flood       0.77      0.71      0.74      1111
          Landslide       0.31      0.39      0.34       155
              Storm       0.74      0.74      0.74       899
  Volcanic activity       0.55      0.49      0.52        53
           Wildfire       0.47      0.43      0.45        94

           accuracy                           0.71      2896
          macro avg       0.62      0.62      0.62      2896
       weighted avg       0.72      0.71      0.71      2896



In [13]:
# Regressor before/after using the promoted feature/target set per target
reg_rows = []
for key in ALL_TARGETS:
    base_m = eval_target(key, BASE_FEATURES, tgt_train_nom, tgt_test_nom)
    win_cfg = best[key]
    feats = BASE_FEATURES + ([] if win_cfg.startswith("baseline") else
            (["income_group"] if "income" in win_cfg else []) +
            (["income_group","log_population"] if "population" in win_cfg else []))
    feats = list(dict.fromkeys(feats))  # dedupe, keep order
    money = "nom" if win_cfg.startswith("baseline") else "cpi"
    tr, te = pick(money)
    win_m = eval_target(key, feats, tr, te)
    reg_rows.append({"target":key, "baseline_err":round(base_m["err_factor"],2),
                     "boosted_err":round(win_m["err_factor"],2), "won_with":win_cfg,
                     "baseline_R2log":round(base_m["R2_log"],3), "boosted_R2log":round(win_m["R2_log"],3)})
print("IMPACT REGRESSORS — before vs after (err_factor = typical x-error, lower is better)")
display(pd.DataFrame(reg_rows).set_index("target"))

IMPACT REGRESSORS — before vs after (err_factor = typical x-error, lower is better)


,baseline_err,boosted_err,won_with,baseline_R2log,boosted_R2log
target,,,,,
deaths,2.97,2.97,+income,0.338,0.338
injuries,3.98,3.93,+population,0.206,0.215
affected,5.72,5.62,+income,0.404,0.418
damage,6.04,5.81,+income,0.218,0.216
uninsured,11.81,11.81,"baseline (16, nominal)",-0.065,-0.065


## 11 · Summary — honest before/after

In [14]:
b = boosted.loc["nb04 baseline (16, fixed)"]; a = boosted.loc["BOOSTED (final)"]
print("CLASSIFIER (held-out, same split as nb04):")
print(f"  Macro F1     {b['f1_macro']:.4f}  ->  {a['f1_macro']:.4f}   (delta {a['f1_macro']-b['f1_macro']:+.4f})")
print(f"  Weighted F1  {b['f1_weighted']:.4f}  ->  {a['f1_weighted']:.4f}   (delta {a['f1_weighted']-b['f1_weighted']:+.4f})")
print(f"  Accuracy     {b['accuracy']:.4f}  ->  {a['accuracy']:.4f}")
print(f"  what moved it: {'Optuna tuning' if RUN_OPTUNA else 'features only'}"
      f"{' + geo features' if USE_GEO else ''}")
print()
print("IMPACT REGRESSORS (err_factor, lower better):")
for r in reg_rows:
    tag = "WIN" if r["boosted_err"] < r["baseline_err"] else "no gain"
    print(f"  {r['target']:<10} {r['baseline_err']:5.2f}x -> {r['boosted_err']:5.2f}x  [{tag:<7}] via {r['won_with']}")
print()
print("Only features that lowered error were promoted. Anything marked 'no gain' is NOT recommended for")
print("production — same validate-first discipline as the project's SRTM / frac_* / threshold rollbacks.")

CLASSIFIER (held-out, same split as nb04):
  Macro F1     0.6024  ->  0.6164   (delta +0.0140)
  Weighted F1  0.7077  ->  0.7129   (delta +0.0052)
  Accuracy     0.7030  ->  0.7068
  what moved it: Optuna tuning + geo features

IMPACT REGRESSORS (err_factor, lower better):
  deaths      2.97x ->  2.97x  [no gain] via +income
  injuries    3.98x ->  3.93x  [WIN    ] via +population
  affected    5.72x ->  5.62x  [WIN    ] via +income
  damage      6.04x ->  5.81x  [WIN    ] via +income
  uninsured  11.81x -> 11.81x  [no gain] via baseline (16, nominal)

Only features that lowered error were promoted. Anything marked 'no gain' is NOT recommended for
production — same validate-first discipline as the project's SRTM / frac_* / threshold rollbacks.
